# Model 3 — 06/07 Validation Quality Gate

## Why this notebook exists

The first external-validation notebook produced a very strong binary F1 but only `DDOS` and `PORTSCAN` coverage, with a low attack-family macro F1. That is **not enough evidence to approve Model 3**.

This notebook performs a stricter **file-grouped validation experiment**:

- a complete CSV file is held out from model training;
- the candidate model is trained only on the remaining files;
- the held-out file is used exclusively for evaluation;
- binary detection and attack-family classification are measured separately;
- every original label in the held-out file is reported;
- this experiment is a generalization test, not a production release.

Do not overwrite the current production artifact from this notebook. If the results are acceptable, retrain the final production model only after the validation design is approved.

In [1]:

from pathlib import Path
import json, time, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "EVIDENT" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT/"data"/"raw"/"behaviour"
META_DIR = PROJECT_ROOT/"models"/"behavior"/"metadata"
META_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(DATA_DIR.glob("*.csv"))
print("Files:")
for i,p in enumerate(files):
    print(i, p.name)

assert len(files) >= 2, "Need at least two CSV files for grouped validation."


Files:
0 Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
1 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
2 Friday-WorkingHours-Morning.pcap_ISCX.csv
3 Monday-WorkingHours.pcap_ISCX.csv
4 Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
5 Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
6 Tuesday-WorkingHours.pcap_ISCX.csv
7 Wednesday-workingHours.pcap_ISCX.csv


In [2]:

# Choose one complete file as an untouched validation group.
# Change this only if you have a specific experimental reason.
HOLDOUT_FILE = files[0]

print("HOLDOUT FILE:")
print(HOLDOUT_FILE.name)

# The remaining files are the candidate training groups.
TRAIN_FILES = [p for p in files if p != HOLDOUT_FILE]
print("\nTraining groups:")
for p in TRAIN_FILES:
    print(" -", p.name)


HOLDOUT FILE:
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Training groups:
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [3]:

def clean_col(c):
    return " ".join(str(c).strip().replace("\ufeff","").split())

def norm(x):
    return "UNKNOWN" if pd.isna(x) else str(x).strip().upper().replace(" ","_")

def family(x):
    x = norm(x)
    if x == "BENIGN": return "BENIGN"
    if "PATATOR" in x or "BRUTE" in x: return "BRUTE_FORCE"
    if "PORTSCAN" in x or "PORT_SCAN" in x: return "PORT_SCAN"
    if "DDOS" in x or x.startswith("DOS"): return "DOS_DDOS"
    if "WEB_ATTACK" in x or "SQL_INJECTION" in x or "XSS" in x: return "WEB_ATTACK"
    if "BOT" in x: return "BOTNET"
    if "INFILTRATION" in x: return "INFILTRATION"
    if "HEARTBLEED" in x: return "HEARTBLEED"
    return "OTHER_ATTACK"

DROP = {
    "Flow ID","Source IP","Src IP","Destination IP","Dst IP",
    "Timestamp","Source Port","Src Port",
    "Destination Port","Dst Port","Protocol"
}

CAP_PER_LABEL = 50000
CHUNK_SIZE = 100000

def load_group(path, cap_per_label=CAP_PER_LABEL):
    parts = []
    counts = {}
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, low_memory=False):
        chunk.columns = [clean_col(c) for c in chunk.columns]
        chunk["Label"] = chunk["Label"].map(norm)
        selected = []
        for label, part in chunk.groupby("Label", sort=False):
            used = counts.get(label, 0)
            remaining = cap_per_label - used
            if remaining <= 0:
                continue
            n = min(remaining, len(part))
            selected.append(part.iloc[:n])
            counts[label] = used + n
        if selected:
            parts.append(pd.concat(selected, ignore_index=True))
        if all(v >= cap_per_label for v in counts.values()) and len(counts) > 0:
            # Continue only if there could be a label not seen yet; otherwise this is enough.
            pass
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

def prepare(df):
    df = df.copy()
    df["is_attack"] = (df["Label"] != "BENIGN").astype(int)
    df["attack_family"] = df["Label"].map(family)

    feature_cols = [
        c for c in df.columns
        if c not in DROP and c not in {"Label","is_attack","attack_family"}
    ]

    X = df[feature_cols].copy()
    for c in feature_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    X = X.replace([np.inf,-np.inf], np.nan)

    # Remove completely missing / constant columns for this experiment.
    feature_cols = [
        c for c in feature_cols
        if X[c].notna().any() and X[c].nunique(dropna=True) > 1
    ]
    return X[feature_cols], df[["Label","is_attack","attack_family"]], feature_cols


In [4]:

train_parts = []
for path in TRAIN_FILES:
    print("Loading training group:", path.name)
    d = load_group(path)
    print(" rows:", len(d), "labels:", d["Label"].nunique())
    train_parts.append(d)

train_df = pd.concat(train_parts, ignore_index=True)
holdout_df = load_group(HOLDOUT_FILE)

print("\nTRAIN LABELS")
print(train_df["Label"].value_counts())
print("\nHOLDOUT LABELS")
print(holdout_df["Label"].value_counts())

X_train, y_train, feature_cols = prepare(train_df)
X_holdout, y_holdout, holdout_features = prepare(holdout_df)

# Align holdout strictly to training feature schema.
X_holdout = X_holdout.reindex(columns=feature_cols)

print("\nTraining rows:", len(X_train))
print("Holdout rows:", len(X_holdout))
print("Features:", len(feature_cols))
print("Holdout families:", sorted(y_holdout["attack_family"].unique()))


Loading training group: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 rows: 100000 labels: 2
Loading training group: Friday-WorkingHours-Morning.pcap_ISCX.csv
 rows: 51966 labels: 2
Loading training group: Monday-WorkingHours.pcap_ISCX.csv
 rows: 50000 labels: 1
Loading training group: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 rows: 50036 labels: 2
Loading training group: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 rows: 52180 labels: 4
Loading training group: Tuesday-WorkingHours.pcap_ISCX.csv
 rows: 63835 labels: 3
Loading training group: Wednesday-workingHours.pcap_ISCX.csv
 rows: 121599 labels: 6

TRAIN LABELS
Label
BENIGN                        350000
PORTSCAN                       50000
DOS_HULK                       50000
DOS_GOLDENEYE                  10293
FTP-PATATOR                     7938
SSH-PATATOR                     5897
DOS_SLOWLORIS                   5796
DOS_SLOWHTTPTEST                5499
BOT                             1966


In [5]:

def make_model(iterations):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", HistGradientBoostingClassifier(
            max_iter=iterations,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42
        ))
    ])

binary_model = make_model(250)
family_model = make_model(300)

print("Training binary detector on NON-HOLDOUT files...")
binary_model.fit(X_train, y_train["is_attack"])

attack_train = y_train["attack_family"] != "BENIGN"
print("Training attack-family detector...")
family_model.fit(X_train.loc[attack_train], y_train.loc[attack_train,"attack_family"])

print("Grouped validation models trained.")


Training binary detector on NON-HOLDOUT files...
Training attack-family detector...
Grouped validation models trained.


In [6]:

# Binary evaluation
start = time.perf_counter()
yb_pred = binary_model.predict(X_holdout)
elapsed = (time.perf_counter()-start)*1000/max(len(X_holdout),1)

p,r,f,_ = precision_recall_fscore_support(
    y_holdout["is_attack"], yb_pred,
    average="binary", zero_division=0
)

binary = {
    "accuracy": float(accuracy_score(y_holdout["is_attack"], yb_pred)),
    "precision": float(p),
    "recall": float(r),
    "f1": float(f),
    "confusion_matrix": confusion_matrix(
        y_holdout["is_attack"], yb_pred
    ).tolist(),
    "classification_report": classification_report(
        y_holdout["is_attack"], yb_pred,
        output_dict=True, zero_division=0
    )
}

print(json.dumps(binary, indent=2))


{
  "accuracy": 0.82465,
  "precision": 0.9844220955564177,
  "recall": 0.65974,
  "f1": 0.7900226322911303,
  "confusion_matrix": [
    [
      49478,
      522
    ],
    [
      17013,
      32987
    ]
  ],
  "classification_report": {
    "0": {
      "precision": 0.7441307846174671,
      "recall": 0.98956,
      "f1-score": 0.8494733498725223,
      "support": 50000.0
    },
    "1": {
      "precision": 0.9844220955564177,
      "recall": 0.65974,
      "f1-score": 0.7900226322911303,
      "support": 50000.0
    },
    "accuracy": 0.82465,
    "macro avg": {
      "precision": 0.8642764400869424,
      "recall": 0.82465,
      "f1-score": 0.8197479910818264,
      "support": 100000.0
    },
    "weighted avg": {
      "precision": 0.8642764400869423,
      "recall": 0.82465,
      "f1-score": 0.8197479910818263,
      "support": 100000.0
    }
  }
}


In [7]:

# Attack-family evaluation
attack_mask = y_holdout["attack_family"] != "BENIGN"

if attack_mask.any():
    yf_true = y_holdout.loc[attack_mask,"attack_family"]
    yf_pred = family_model.predict(X_holdout.loc[attack_mask])

    fp, fr, ff, _ = precision_recall_fscore_support(
        yf_true, yf_pred, average="macro", zero_division=0
    )

    family_metrics = {
        "macro_f1": float(ff),
        "weighted_f1": float(
            precision_recall_fscore_support(
                yf_true, yf_pred, average="weighted", zero_division=0
            )[2]
        ),
        "accuracy": float(accuracy_score(yf_true, yf_pred)),
        "labels": sorted(yf_true.unique().tolist()),
        "classification_report": classification_report(
            yf_true, yf_pred, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            yf_true, yf_pred,
            labels=sorted(yf_true.unique())
        ).tolist()
    }
else:
    family_metrics = None

print(json.dumps(family_metrics, indent=2))


{
  "macro_f1": 0.3312132678528299,
  "weighted_f1": 0.9936398035584897,
  "accuracy": 0.98736,
  "labels": [
    "DOS_DDOS"
  ],
  "classification_report": {
    "BOTNET": {
      "precision": 0.0,
      "recall": 0.0,
      "f1-score": 0.0,
      "support": 0.0
    },
    "DOS_DDOS": {
      "precision": 1.0,
      "recall": 0.98736,
      "f1-score": 0.9936398035584897,
      "support": 50000.0
    },
    "INFILTRATION": {
      "precision": 0.0,
      "recall": 0.0,
      "f1-score": 0.0,
      "support": 0.0
    },
    "accuracy": 0.98736,
    "macro avg": {
      "precision": 0.3333333333333333,
      "recall": 0.32912,
      "f1-score": 0.3312132678528299,
      "support": 50000.0
    },
    "weighted avg": {
      "precision": 1.0,
      "recall": 0.98736,
      "f1-score": 0.9936398035584897,
      "support": 50000.0
    }
  },
  "confusion_matrix": [
    [
      49368
    ]
  ]
}


In [8]:

result = {
    "validation_type": "leave-one-file-group-out",
    "holdout_file": HOLDOUT_FILE.name,
    "training_files": [p.name for p in TRAIN_FILES],
    "training_rows": int(len(X_train)),
    "holdout_rows": int(len(X_holdout)),
    "binary": binary,
    "attack_family": family_metrics,
    "latency": {
        "average_binary_prediction_ms": float(elapsed)
    },
    "coverage": {
        "holdout_original_labels": sorted(y_holdout["Label"].unique().tolist()),
        "holdout_attack_families": sorted(
            y_holdout.loc[y_holdout["attack_family"]!="BENIGN","attack_family"].unique().tolist()
        )
    }
}

out = META_DIR/"grouped_external_validation_metrics.json"
out.write_text(json.dumps(result, indent=2), encoding="utf-8")

print("\n=== QUALITY GATE RESULT ===")
print(json.dumps({
    "holdout": HOLDOUT_FILE.name,
    "binary_f1": binary["f1"],
    "family_macro_f1": family_metrics["macro_f1"] if family_metrics else None,
    "coverage": result["coverage"]
}, indent=2))
print("\nSaved:", out)



=== QUALITY GATE RESULT ===
{
  "holdout": "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
  "binary_f1": 0.7900226322911303,
  "family_macro_f1": 0.3312132678528299,
  "coverage": {
    "holdout_original_labels": [
      "BENIGN",
      "DDOS"
    ],
    "holdout_attack_families": [
      "DOS_DDOS"
    ]
  }
}

Saved: c:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\metadata\grouped_external_validation_metrics.json


## Interpretation rule

Do **not** approve Model 3 from a single grouped split.

Repeat this notebook for multiple holdout files/groups, for example:

- Friday
- Thursday
- Tuesday
- Wednesday

and compare the results.

A strong binary F1 with poor attack-family macro F1 means the model can distinguish **attack vs benign**, but cannot reliably distinguish attack categories. That is a real quality finding and should be fixed before production if attack-family classification is part of the demo.

The current production artifact should remain untouched until the grouped validation experiments are satisfactory.